# Alloy no-monitoring assessment

Assesses Alloy **without using the LGTM stack as an instrument** -- kube API state,
events, PVCs, and direct `/-/ready` probes only. Alloy is the most foundational collector:
when it is down there are *no* metrics to read at all, so the kube-API-only view is the
honest fallback. Normal-times status: [alloy-health.ipynb](alloy-health.ipynb).

Design: [tiles#651](https://github.com/symmatree/tiles/issues/651).

Run from `notebooks/`: `jupyter nbconvert --to notebook --execute --inplace alloy-nomon.ipynb`

In [1]:
namespace = "alloy"
tenant = "tiles"
ready_port = 12345     # Alloy HTTP port exposing /-/ready
capture_file = ""      # replay a prior raw capture JSON instead of querying live
output_dir = ""        # if set: write raw capture + agent stats there
debug = False

In [2]:
import json
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import nb_capture as nbc

NOTEBOOK = 'alloy-nomon'
cap = nbc.Capture(replay_file=capture_file, namespace=namespace, tenant=tenant, ready_port=ready_port)
namespace, tenant = cap.meta['namespace'], cap.meta['tenant']
ready_port = cap.meta['ready_port']
kube = nbc.Kube(cap)
print(f"namespace: {namespace}  {'REPLAY of ' + cap.run_at if cap.replay else 'live'}")

namespace: alloy  live


In [3]:
# --- ASSUMPTIONS: only that the kube API answers (Kube.get raises otherwise). ---
ns = kube.get('namespace', 'namespace', namespace)
print(f"kube API: ok; namespace {namespace} is {ns['status']['phase']}")

kube API: ok; namespace alloy is Active


In [4]:
# --- PODS: phase, readiness, restarts, last terminations, CrashLoop (kube API view) ---
pods_raw = kube.get('pods', 'pods', '-n', namespace)
rows = []
for p in pods_raw['items']:
    cs = p['status'].get('containerStatuses', [])
    term = next(((c['lastState'].get('terminated') or {}) for c in cs if c['lastState'].get('terminated')), {})
    waiting = next((c['state']['waiting']['reason'] for c in cs
                    if c.get('state', {}).get('waiting', {}).get('reason')), '')
    rows.append({'pod': p['metadata']['name'],
                 'phase': p['status'].get('phase', '?'),
                 'ready': f"{sum(c['ready'] for c in cs)}/{len(cs)}",
                 'restarts': sum(c['restartCount'] for c in cs),
                 'waiting': waiting,
                 'last_term': f"{term.get('reason', '')} {(term.get('finishedAt') or '')[:16]}".strip()})
pods_df = pd.DataFrame(rows).set_index('pod').sort_values('restarts', ascending=False)
ready_n = pods_df['ready'].str.split('/', expand=True)
not_ready = sorted(pods_df.index[(pods_df['phase'] != 'Running') | (ready_n[0] != ready_n[1])])
crashloopers = sorted(pods_df.index[pods_df['waiting'] == 'CrashLoopBackOff'])
print(pods_df.to_string())
print(f"not ready: {not_ready or 'none'}")
print(f"CrashLoopBackOff: {crashloopers or 'none'}")

                                             phase ready  restarts waiting               last_term
pod                                                                                               
alloy-alloy-singleton-84894dfdb8-pxs2s      Failed   0/2       160          Error 2026-07-25T14:32
alloy-kube-state-metrics-5fd87c5d6f-mv22j   Failed   0/1        57          Error 2026-07-25T14:59
alloy-kube-state-metrics-65c585cc7-vnqkc   Running   1/1         2          Error 2026-07-29T18:11
alloy-alloy-logs-77l89                     Running   2/2         0                                
alloy-alloy-logs-2zncm                     Running   2/2         0                                
alloy-alloy-logs-6nppw                     Running   2/2         0                                
alloy-alloy-logs-6qhkq                     Running   2/2         0                                
alloy-alloy-logs-skrkl                     Running   2/2         0                                
alloy-allo

In [5]:
# --- EVENTS: Warning events in the namespace (whatever etcd still holds) ---
ev = kube.get('events', 'events', '-n', namespace)

def ev_time(e):
    return e.get('lastTimestamp') or e.get('eventTime') or e['metadata']['creationTimestamp']

warn = sorted((e for e in ev['items'] if e.get('type') == 'Warning'), key=ev_time, reverse=True)
warn_reasons = dict(Counter(e.get('reason', '?') for e in warn))
print(f"{len(ev['items'])} events retained, {len(warn)} warnings; by reason: {warn_reasons or 'none'}")
for e in warn[:15]:
    print(f"  {ev_time(e)[:19]} x{e.get('count') or 1:<3} {e.get('reason', '?'):<18} "
          f"{e['involvedObject'].get('name', '')[:40]:<40} {e.get('message', '')[:90]}")

322 events retained, 24 warnings; by reason: {'Unhealthy': 13, 'NodeNotReady': 6, 'BackOff': 5}
  2026-07-29T18:13:25 x1   Unhealthy          alloy-alloy-logs-vptfn                   Readiness probe failed: Get "http://10.0.147.141:12345/-/ready": dial tcp 10.0.147.141:123
  2026-07-29T18:11:36 x2   NodeNotReady       alloy-alloy-metrics-0                    Node is not ready
  2026-07-29T18:11:36 x2   NodeNotReady       alloy-alloy-singleton-75875cfbc5-vkxqw   Node is not ready
  2026-07-29T18:11:34 x2   NodeNotReady       alloy-alloy-receiver-594f6bb9-9j8fl      Node is not ready
  2026-07-29T18:11:32 x2   NodeNotReady       alloy-node-exporter-n9ws9                Node is not ready
  2026-07-29T18:11:22 x6   Unhealthy          alloy-kube-state-metrics-65c585cc7-vnqkc Liveness probe failed: HTTP probe failed with statuscode: 503
  2026-07-29T18:11:19 x1   NodeNotReady       alloy-alloy-logs-fw8r5                   Node is not ready
  2026-07-29T18:11:19 x1   NodeNotReady       alloy-

In [6]:
# --- DIRECT /-/ready PROBES: each Alloy service on its HTTP port, no metrics involved.
# A service whose pods are all down has no endpoints -> the probe errors, which is the
# point: it distinguishes a crashlooping component from a merely-degraded one.
svcs = kube.get('services', 'services', '-n', namespace)
probe = {}
for s in svcs['items']:
    nm = s['metadata']['name']
    if not any(p['port'] == ready_port for p in s['spec']['ports']):
        continue
    rec = cap.http(f'ready: {nm}', f'http://{nm}.{namespace}.svc:{ready_port}/-/ready', timeout=5)
    body = (rec.get('text') or json.dumps(rec.get('json', ''))).strip()[:40]
    probe[nm] = rec.get('error', '').split(':')[0] or f"{rec.get('status')} {body}"
    print(f"  {nm:<30} {probe[nm]}")
ready_failed = sorted(nm for nm, r in probe.items() if not str(r).startswith('200'))
print(f"failed probes: {ready_failed or 'none'}")

  alloy-alloy-logs               200 Alloy is ready.


  alloy-alloy-metrics            200 Alloy is ready.
  alloy-alloy-metrics-cluster    200 Alloy is ready.
  alloy-alloy-receiver           200 Alloy is ready.
  alloy-alloy-singleton          200 Alloy is ready.
failed probes: none


In [7]:
# --- PVCs (Alloy metrics WAL is emptyDir here, so typically none) ---
pvcs = kube.get('pvcs', 'pvc', '-n', namespace)
pvc_phase = {p['metadata']['name']: p['status']['phase'] for p in pvcs['items']}
pvc_not_bound = sorted(n for n, ph in pvc_phase.items() if ph != 'Bound')
print(f"{len(pvc_phase)} PVCs: {pvc_phase or 'none'}")
print(f"not bound: {pvc_not_bound or 'none'}")

0 PVCs: none
not bound: none


In [8]:
# --- SUMMARY ---
findings = []
if crashloopers:
    findings.append(f"pods in CrashLoopBackOff: {crashloopers}")
if not_ready:
    findings.append(f"pods not ready: {not_ready}")
if ready_failed:
    findings.append(f"/-/ready probes failing (no endpoints or unready): {ready_failed}")
if pvc_not_bound:
    findings.append(f"PVCs not bound: {pvc_not_bound}")
bad_reasons = {r: n for r, n in warn_reasons.items()
               if r in ('OOMKilling', 'BackOff', 'CrashLoopBackOff', 'FailedScheduling',
                        'FailedMount', 'Unhealthy', 'Evicted')}
if bad_reasons:
    findings.append(f"warning events of concern: {bad_reasons}")

print("=" * 60)
print("ALLOY NO-MONITORING SUMMARY")
print("=" * 60)
print(f"run:      {cap.run_at}  ({cap.mode})")
print(f"scope:    namespace {namespace}, kube API + direct /-/ready only")
print()
if findings:
    print("Findings:")
    for f in findings:
        print(f"  ! {f}")
else:
    print(f"Findings: none -- {len(pods_df)} pods running/ready, {len(probe)} probes green")

agent_stats = {
    'notebook': f'{NOTEBOOK}.ipynb',
    'run_at': cap.run_at,
    'mode': cap.mode,
    'namespace': namespace,
    'findings': findings,
    'pods': {'total': len(pods_df), 'not_ready': not_ready, 'crashloop': crashloopers,
             'restarts_by_pod': {p: int(n) for p, n in pods_df['restarts'].items() if n > 0},
             'last_term_by_pod': {p: t for p, t in pods_df['last_term'].items() if t}},
    'events': {'warning_count': len(warn), 'warning_reasons': warn_reasons},
    'probes': probe,
    'ready_failed': ready_failed,
    'pvcs': {'phase': pvc_phase, 'not_bound': pvc_not_bound},
}
print()
print(json.dumps(agent_stats, indent=1))

if output_dir:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    cap.save(out / f'{NOTEBOOK}.capture.json')
    (out / f'{NOTEBOOK}.stats.json').write_text(json.dumps(agent_stats, indent=2))
    print(f"wrote {out / f'{NOTEBOOK}.capture.json'} and {out / f'{NOTEBOOK}.stats.json'}")

ALLOY NO-MONITORING SUMMARY
run:      2026-07-29T18:36:29.696912+00:00  (live)
scope:    namespace alloy, kube API + direct /-/ready only

Findings:
  ! pods not ready: ['alloy-alloy-singleton-84894dfdb8-pxs2s', 'alloy-kube-state-metrics-5fd87c5d6f-mv22j']
  ! warning events of concern: {'Unhealthy': 13, 'BackOff': 5}

{
 "notebook": "alloy-nomon.ipynb",
 "run_at": "2026-07-29T18:36:29.696912+00:00",
 "mode": "live",
 "namespace": "alloy",
 "findings": [
  "pods not ready: ['alloy-alloy-singleton-84894dfdb8-pxs2s', 'alloy-kube-state-metrics-5fd87c5d6f-mv22j']",
  "warning events of concern: {'Unhealthy': 13, 'BackOff': 5}"
 ],
 "pods": {
  "total": 23,
  "not_ready": [
   "alloy-alloy-singleton-84894dfdb8-pxs2s",
   "alloy-kube-state-metrics-5fd87c5d6f-mv22j"
  ],
  "crashloop": [],
  "restarts_by_pod": {
   "alloy-alloy-singleton-84894dfdb8-pxs2s": 160,
   "alloy-kube-state-metrics-5fd87c5d6f-mv22j": 57,
   "alloy-kube-state-metrics-65c585cc7-vnqkc": 2
  },
  "last_term_by_pod": {
   